# 🔹 Notebook 4 — Entity Extraction con Ollama

**Obiettivo**: Estrarre entita' e relazioni da testo usando un LLM locale via Ollama.

Questo notebook implementa la **Fase 5 della pipeline di ingestion** del libro.


## Setup

In [ ]:
import os, json
import httpx

OLLAMA_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
LLM_MODEL = os.getenv('OLLAMA_LLM_MODEL', 'qwen2.5:14b')

def chat_json(prompt, system):
    resp = httpx.post(f'{OLLAMA_URL}/api/chat', json={
        'model': LLM_MODEL, 'stream': False, 'format': 'json',
        'messages': [{'role':'system','content':system}, {'role':'user','content':prompt}]
    }, timeout=120)
    return json.loads(resp.json()['message']['content'])

print(f'LLM: {LLM_MODEL} @ {OLLAMA_URL}')

## 4.1 — Estrazione entita' e relazioni

In [ ]:
EXTRACTION_SYSTEM = """Sei un estrattore di Knowledge Graph. Dato un testo, estrai:
- entities: lista di {name, type, description}
  type puo' essere: Person, Organization, Technology, Product, Process, Location, Concept
- relations: lista di {source, target, type, confidence}
  type puo' essere: WORKS_AT, USES, CREATED_BY, LOCATED_IN, RELATED_TO, DEPENDS_ON, PART_OF

Rispondi SOLO in JSON valido con chiavi 'entities' e 'relations'."""

sample_text = """
Alice Smith e' una Senior Engineer presso TechCorp a San Francisco.
Lavora dal 2020 su progetti di Knowledge Graph usando Neo4j e Python.
Collabora con Bob Jones, Data Engineer nello stesso team, che si occupa
della pipeline Redis per il vector store. Il progetto principale e'
la piattaforma KG-Platform, che integra ricerca semantica e RAG.
"""

result = chat_json(sample_text, EXTRACTION_SYSTEM)
print(json.dumps(result, indent=2, ensure_ascii=False))

## 4.2 — Visualizza il grafo estratto

In [ ]:
print('=== ENTITA ESTRATTE ===')
for e in result.get('entities', []):
    print(f"  [{e.get('type', '?'):15s}] {e.get('name', '?')}")
    if e.get('description'):
        print(f"{'':19s}{e['description']}")

print('\n=== RELAZIONI ESTRATTE ===')
for rel in result.get('relations', []):
    conf = rel.get('confidence', 0)
    bar = '█' * int(conf * 10) if isinstance(conf, (int, float)) else '???'
    print(f"  {rel.get('source','?')} --[{rel.get('type','?')}]--> {rel.get('target','?')}  {bar}")

## 4.3 — Carica entita' estratte in Neo4j

In [ ]:
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
driver = GraphDatabase.driver(NEO4J_URI, auth=('neo4j', os.getenv('NEO4J_PASSWORD','yourpassword')))

with driver.session() as session:
    # Upsert entita'
    for e in result.get('entities', []):
        session.run(
            f"MERGE (n:{e.get('type','Entity')} {{name: $name}}) "
            f"SET n.description = $desc, n.extracted = true",
            name=e.get('name',''), desc=e.get('description','')
        )
    
    # Upsert relazioni
    for rel in result.get('relations', []):
        rtype = rel.get('type', 'RELATED_TO').replace(' ', '_').upper()
        session.run(
            f"MATCH (a {{name: $src}}), (b {{name: $tgt}}) "
            f"MERGE (a)-[r:{rtype}]->(b) "
            f"SET r.confidence = $conf, r.extracted = true",
            src=rel.get('source',''), tgt=rel.get('target',''),
            conf=rel.get('confidence', 0.5)
        )
    
    # Conta nodi estratti
    count = session.run('MATCH (n {extracted: true}) RETURN count(n) AS c').single()['c']
    print(f'Nodi estratti e caricati in Neo4j: {count}')

driver.close()